In [44]:
#30. A query on a trillion-row table is slow — how do you optimize it? (partitioning, clustering, predicate pushdown, avoiding SELECT *)


In [45]:
# SQL Theory & Optimization
# A query on a trillion-row table is slow — how do you optimize it? (partitioning, clustering, predicate pushdown, avoiding SELECT *)
# Difference between WHERE and HAVING; when does each execute?
# Difference between UNION and UNION ALL (and performance implications).
# Explain query execution order (FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY).
# What are indexes? When do they hurt performance? Why does BigQuery not use traditional indexes?
# Correlated vs non-correlated subqueries — performance impact.
# CTEs vs temp tables vs subqueries — when to use which.
# How would you find and handle data skew in a JOIN on a distributed engine?


In [46]:
from pyspark.sql import SparkSession

from delta import configure_spark_with_delta_pip

# spark.stop()
builder = (

    SparkSession.builder

    .appName("DeltaLocal")

    .master("local[1]")

    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")

    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
# from pyspark.sql import SparkSession

# spark = (
#     SparkSession.builder
#     .appName("LocalSparkTest")
#     .master("local[1]")
#     .config("spark.driver.host", "127.0.0.1")
#     .config("spark.driver.bindAddress", "127.0.0.1")
#     .config("spark.ui.enabled", "false")
#     .config("spark.sql.shuffle.partitions", "1")
#     .getOrCreate()
# )
def runsql(query):
    spark.sql(query).show(truncate=False)

In [3]:
from pyspark.sql.functions import col, to_date, to_timestamp

data = [
    (1, "C001", "P001", "Bangkok", "2026-01-01", 100, "completed", "2026-01-01 10:00:00"),
    (2, "C002", "P002", "Bangkok", "2026-01-02", 200, "completed", "2026-01-02 11:00:00"),
    (3, "C003", "P003", "Chiang Mai", "2026-01-03", 300, "cancelled", "2026-01-03 12:00:00"),
    (4, "C004", "P001", "Bangkok", "2026-02-01", 400, "completed", "2026-02-01 13:00:00"),
    (5, "C005", "P002", "Phuket", "2026-02-05", 500, "completed", "2026-02-05 14:00:00"),
    (6, "C006", "P003", "Bangkok", "2025-12-30", 600, "completed", "2025-12-30 15:00:00"),
    (7, "C007", "P001", "Bangkok", "2026-03-10", 700, "completed", "2026-03-10 16:00:00"),
    (8, "C008", "P004", "Chiang Mai", "2026-03-11", 800, "completed", "2026-03-11 17:00:00")
]

columns = [
    "order_id",
    "customer_id",
    "product_id",
    "region",
    "order_date",
    "amount",
    "status",
    "created_at"
]

orders_df = spark.createDataFrame(data, columns)

orders_df = (
    orders_df
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("created_at", to_timestamp(col("created_at")))
)

orders_df.createOrReplaceTempView("sales_orders")

orders_df.show(truncate=False)

+--------+-----------+----------+----------+----------+------+---------+-------------------+
|order_id|customer_id|product_id|region    |order_date|amount|status   |created_at         |
+--------+-----------+----------+----------+----------+------+---------+-------------------+
|1       |C001       |P001      |Bangkok   |2026-01-01|100   |completed|2026-01-01 10:00:00|
|2       |C002       |P002      |Bangkok   |2026-01-02|200   |completed|2026-01-02 11:00:00|
|3       |C003       |P003      |Chiang Mai|2026-01-03|300   |cancelled|2026-01-03 12:00:00|
|4       |C004       |P001      |Bangkok   |2026-02-01|400   |completed|2026-02-01 13:00:00|
|5       |C005       |P002      |Phuket    |2026-02-05|500   |completed|2026-02-05 14:00:00|
|6       |C006       |P003      |Bangkok   |2025-12-30|600   |completed|2025-12-30 15:00:00|
|7       |C007       |P001      |Bangkok   |2026-03-10|700   |completed|2026-03-10 16:00:00|
|8       |C008       |P004      |Chiang Mai|2026-03-11|800   |complete

In [4]:
# 2. Slow query example


In [5]:
slow_df = spark.sql("""
SELECT *
FROM sales_orders
WHERE YEAR(order_date) = 2026
  AND UPPER(region) = 'BANGKOK'
  AND status = 'completed'
ORDER BY created_at
""")

slow_df.show(truncate=False)

+--------+-----------+----------+-------+----------+------+---------+-------------------+
|order_id|customer_id|product_id|region |order_date|amount|status   |created_at         |
+--------+-----------+----------+-------+----------+------+---------+-------------------+
|1       |C001       |P001      |Bangkok|2026-01-01|100   |completed|2026-01-01 10:00:00|
|2       |C002       |P002      |Bangkok|2026-01-02|200   |completed|2026-01-02 11:00:00|
|4       |C004       |P001      |Bangkok|2026-02-01|400   |completed|2026-02-01 13:00:00|
|7       |C007       |P001      |Bangkok|2026-03-10|700   |completed|2026-03-10 16:00:00|
+--------+-----------+----------+-------+----------+------+---------+-------------------+



In [6]:
# 1. SELECT * reads all columns
# 2. YEAR(order_date) applies function on partition column
# 3. UPPER(region) applies function on filter column
# 4. ORDER BY created_at causes global sort
# 5. It may scan more data than needed

In [7]:
# 3. Optimized query

In [8]:
optimized_df = spark.sql("""
SELECT
    order_id,
    customer_id,
    product_id,
    region,
    order_date,
    amount
FROM sales_orders
WHERE order_date >= DATE '2026-01-01'
  AND order_date <  DATE '2027-01-01'
  AND region = 'Bangkok'
  AND status = 'completed'
""")

optimized_df.show(truncate=False)

+--------+-----------+----------+-------+----------+------+
|order_id|customer_id|product_id|region |order_date|amount|
+--------+-----------+----------+-------+----------+------+
|1       |C001       |P001      |Bangkok|2026-01-01|100   |
|2       |C002       |P002      |Bangkok|2026-01-02|200   |
|4       |C004       |P001      |Bangkok|2026-02-01|400   |
|7       |C007       |P001      |Bangkok|2026-03-10|700   |
+--------+-----------+----------+-------+----------+------+



In [9]:
# 1. No SELECT *
# 2. Direct date range filter allows partition pruning
# 3. No function on region column
# 4. No unnecessary ORDER BY
# 5. Only required columns are read

In [10]:
# 4. Partitioning example

# For a trillion-row table, we usually partition by date.

# Example Delta table creation:

In [11]:
orders_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .save("/tmp/sales_orders_partitioned")

In [17]:


# read partitioned parquet
orders_partitioned_df = spark.read.parquet("/tmp/sales_orders_partitioned")

# create temp view
orders_partitioned_df.createOrReplaceTempView("sales_orders_partitioned")

# query with partition pruning
result_df = spark.sql("""
SELECT
    order_id,
    customer_id,
    product_id,
    region,
    order_date,
    amount
FROM sales_orders_partitioned
WHERE order_date >= DATE '2026-01-01'
  AND order_date <  DATE '2026-02-01'
  AND region = 'Bangkok'
  AND status = 'completed'
""")

result_df.show(truncate=False)

+--------+-----------+----------+-------+----------+------+
|order_id|customer_id|product_id|region |order_date|amount|
+--------+-----------+----------+-------+----------+------+
|2       |C002       |P002      |Bangkok|2026-01-02|200   |
|1       |C001       |P001      |Bangkok|2026-01-01|100   |
+--------+-----------+----------+-------+----------+------+



In [18]:
# 5. Clustering / Z-ordering example

# Partitioning helps reduce date range scan.

# But if most queries also filter by customer_id, product_id, or region, use clustering/Z-ordering.

# In Databricks Delta:

In [22]:
# spark.sql("""
# OPTIMIZE sales_orders_delta
# ZORDER BY (customer_id, product_id, region)
# """)

In [23]:
# 6. Predicate pushdown example

In [24]:
good_pushdown_df = spark.sql("""
SELECT
    order_id,
    customer_id,
    amount
FROM sales_orders
WHERE region = 'Bangkok'
""")

In [25]:
good_pushdown_df.show()

+--------+-----------+------+
|order_id|customer_id|amount|
+--------+-----------+------+
|       1|       C001|   100|
|       2|       C002|   200|
|       4|       C004|   400|
|       6|       C006|   600|
|       7|       C007|   700|
+--------+-----------+------+



In [26]:
# 7. Avoid SELECT *?

In [27]:
good_select_df = spark.sql("""
SELECT
    order_id,
    customer_id,
    amount
FROM sales_orders
WHERE order_date >= DATE '2026-01-01'
  AND order_date < DATE '2026-02-01'
""")

In [29]:
# 8. Filter before join

# Create customer sample:

In [30]:
customers_data = [
    ("C001", "Alice", "Gold"),
    ("C002", "Bob", "Silver"),
    ("C003", "Charlie", "Bronze"),
    ("C004", "David", "Gold"),
    ("C005", "Eva", "Silver")
]

customers_columns = ["customer_id", "customer_name", "segment"]

customers_df = spark.createDataFrame(customers_data, customers_columns)

customers_df.createOrReplaceTempView("customers")

In [31]:
bad_join_df = spark.sql("""
SELECT *
FROM sales_orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.order_date >= DATE '2026-01-01'
  AND o.order_date < DATE '2026-02-01'
""")

In [32]:
better_join_df = spark.sql("""
WITH filtered_orders AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        amount
    FROM sales_orders
    WHERE order_date >= DATE '2026-01-01'
      AND order_date < DATE '2026-02-01'
)

SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    c.segment,
    o.order_date,
    o.amount
FROM filtered_orders o
JOIN customers c
    ON o.customer_id = c.customer_id
""")

better_join_df.show()

+--------+-----------+-------------+-------+----------+------+
|order_id|customer_id|customer_name|segment|order_date|amount|
+--------+-----------+-------------+-------+----------+------+
|       1|       C001|        Alice|   Gold|2026-01-01|   100|
|       2|       C002|          Bob| Silver|2026-01-02|   200|
|       3|       C003|      Charlie| Bronze|2026-01-03|   300|
+--------+-----------+-------------+-------+----------+------+



In [33]:
# 9. Broadcast small dimension table

# If customers is small, broadcast it.?

In [34]:
broadcast_join_df = spark.sql("""
SELECT /*+ BROADCAST(c) */
    o.order_id,
    o.customer_id,
    c.customer_name,
    c.segment,
    o.amount
FROM sales_orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.order_date >= DATE '2026-01-01'
  AND o.order_date < DATE '2026-02-01'
""")

broadcast_join_df.show()

+--------+-----------+-------------+-------+------+
|order_id|customer_id|customer_name|segment|amount|
+--------+-----------+-------------+-------+------+
|       1|       C001|        Alice|   Gold|   100|
|       2|       C002|          Bob| Silver|   200|
|       3|       C003|      Charlie| Bronze|   300|
+--------+-----------+-------------+-------+------+



In [35]:
# 10. Pre-aggregate for dashboard

# If dashboard always asks daily revenue by region, do not scan raw trillion-row table every time.

# Create summary table:

In [36]:
daily_summary_df = spark.sql("""
SELECT
    order_date,
    region,
    SUM(amount) AS total_revenue,
    COUNT(*) AS order_count
FROM sales_orders
WHERE status = 'completed'
GROUP BY
    order_date,
    region
""")

daily_summary_df.createOrReplaceTempView("sales_daily_summary")
daily_summary_df.show()

+----------+----------+-------------+-----------+
|order_date|    region|total_revenue|order_count|
+----------+----------+-------------+-----------+
|2026-01-02|   Bangkok|          200|          1|
|2026-03-11|Chiang Mai|          800|          1|
|2026-03-10|   Bangkok|          700|          1|
|2025-12-30|   Bangkok|          600|          1|
|2026-02-01|   Bangkok|          400|          1|
|2026-01-01|   Bangkok|          100|          1|
|2026-02-05|    Phuket|          500|          1|
+----------+----------+-------------+-----------+



In [37]:
dashboard_df = spark.sql("""
SELECT
    order_date,
    region,
    total_revenue,
    order_count
FROM sales_daily_summary
WHERE order_date >= DATE '2026-01-01'
  AND order_date < DATE '2026-02-01'
  AND region = 'Bangkok'
""")

dashboard_df.show()

+----------+-------+-------------+-----------+
|order_date| region|total_revenue|order_count|
+----------+-------+-------------+-----------+
|2026-01-02|Bangkok|          200|          1|
|2026-01-01|Bangkok|          100|          1|
+----------+-------+-------------+-----------+



In [38]:
# 11. Check query plan

# In Spark:

In [39]:
optimized_df.explain(True)

== Parsed Logical Plan ==
'Project ['order_id, 'customer_id, 'product_id, 'region, 'order_date, 'amount]
+- 'Filter ((('order_date >= 2026-01-01) AND ('order_date < 2027-01-01)) AND (('region = Bangkok) AND ('status = completed)))
   +- 'UnresolvedRelation [sales_orders], [], false

== Analyzed Logical Plan ==
order_id: bigint, customer_id: string, product_id: string, region: string, order_date: date, amount: bigint
Project [order_id#0L, customer_id#1, product_id#2, region#3, order_date#16, amount#5L]
+- Filter (((order_date#16 >= 2026-01-01) AND (order_date#16 < 2027-01-01)) AND ((region#3 = Bangkok) AND (status#6 = completed)))
   +- SubqueryAlias sales_orders
      +- View (`sales_orders`, [order_id#0L,customer_id#1,product_id#2,region#3,order_date#16,amount#5L,status#6,created_at#25])
         +- Project [order_id#0L, customer_id#1, product_id#2, region#3, order_date#16, amount#5L, status#6, to_timestamp(created_at#7, None, TimestampType, Some(Asia/Bangkok), false) AS created_at#25

In [40]:
# 12. Final optimized SQL pattern

In [43]:
final_df = spark.sql("""
WITH filtered_orders AS (
    SELECT
        order_id,
        customer_id,
        product_id,
        order_date,
        amount
    FROM sales_orders
    WHERE order_date >= DATE '2026-01-01'
      AND order_date <  DATE '2026-02-01'
      AND region = 'Bangkok'
      AND status = 'completed'
)

SELECT
    o.order_id,
    o.customer_id,
    o.product_id,
    o.order_date,
    o.amount
FROM filtered_orders o
""")
final_df.show()

+--------+-----------+----------+----------+------+
|order_id|customer_id|product_id|order_date|amount|
+--------+-----------+----------+----------+------+
|       1|       C001|      P001|2026-01-01|   100|
|       2|       C002|      P002|2026-01-02|   200|
+--------+-----------+----------+----------+------+



In [47]:
# For a trillion-row table, I first check the query plan and bytes scanned. Then I reduce scan by using partition pruning with direct date filters, not functions like YEAR(order_date). I avoid SELECT * and select only required columns. I make sure predicates can be pushed down by avoiding functions on filter columns. I use clustering or Z-ordering on common filter and join keys. I filter the large fact table before joining, broadcast small dimensions, and check for data skew. If the query is repeated often, I build an aggregate table or materialized view.

In [48]:
# SQL Theory & Optimization
# A query on a trillion-row table is slow — how do you optimize it? (partitioning, clustering, predicate pushdown, avoiding SELECT *)
# Difference between WHERE and HAVING; when does each execute?
# Difference between UNION and UNION ALL (and performance implications).
# Explain query execution order (FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY).
# What are indexes? When do they hurt performance? Why does BigQuery not use traditional indexes?
# Correlated vs non-correlated subqueries — performance impact.
# CTEs vs temp tables vs subqueries — when to use which.
# How would you find and handle data skew in a JOIN on a distributed engine?


In [49]:
# UNION ALL = faster, keeps duplicates
# UNION     = slower, removes duplicates

In [50]:
sales_jan_data = [
    (1, "C001", 100),
    (2, "C002", 200),
    (3, "C003", 300)
]

sales_feb_data = [
    (3, "C003", 300),
    (4, "C004", 400),
    (5, "C005", 500)
]

columns = ["order_id", "customer_id", "amount"]

sales_jan_df = spark.createDataFrame(sales_jan_data, columns)
sales_feb_df = spark.createDataFrame(sales_feb_data, columns)

sales_jan_df.createOrReplaceTempView("sales_jan")
sales_feb_df.createOrReplaceTempView("sales_feb")

In [51]:
result_df = spark.sql("""
SELECT order_id, customer_id, amount
FROM sales_jan

UNION ALL

SELECT order_id, customer_id, amount
FROM sales_feb
""")

result_df.show()

+--------+-----------+------+
|order_id|customer_id|amount|
+--------+-----------+------+
|       1|       C001|   100|
|       2|       C002|   200|
|       3|       C003|   300|
|       3|       C003|   300|
|       4|       C004|   400|
|       5|       C005|   500|
+--------+-----------+------+



In [52]:
result_df = spark.sql("""
SELECT order_id, customer_id, amount
FROM sales_jan

UNION 

SELECT order_id, customer_id, amount
FROM sales_feb
""")

result_df.show()

+--------+-----------+------+
|order_id|customer_id|amount|
+--------+-----------+------+
|       1|       C001|   100|
|       3|       C003|   300|
|       2|       C002|   200|
|       5|       C005|   500|
|       4|       C004|   400|
+--------+-----------+------+



In [53]:
# # 2. SQL query execution order

# # Theory

# # SQL is written like this:
# SELECT
# FROM
# WHERE
# GROUP BY
# HAVING
# ORDER BY

In [55]:
orders_data = [
    (1, "Bangkok", "completed", 100),
    (2, "Bangkok", "completed", 200),
    (3, "Bangkok", "cancelled", 300),
    (4, "Phuket", "completed", 400),
    (5, "Phuket", "completed", 500),
    (6, "Chiang Mai", "completed", 150)
]

columns = ["order_id", "region", "status", "amount"]

orders_df = spark.createDataFrame(orders_data, columns)
orders_df.createOrReplaceTempView("orders")

In [56]:
orders_data = [
    (1, "Bangkok", "completed", 100),
    (2, "Bangkok", "completed", 200),
    (3, "Bangkok", "cancelled", 300),
    (4, "Phuket", "completed", 400),
    (5, "Phuket", "completed", 500),
    (6, "Chiang Mai", "completed", 150)
]

columns = ["order_id", "region", "status", "amount"]

orders_df = spark.createDataFrame(orders_data, columns)
orders_df.createOrReplaceTempView("orders")

In [57]:
# 1. Table has heavy INSERT / UPDATE / DELETE
# 2. Too many indexes exist
# 3. Indexed column has low selectivity
# 4. Query returns most of the table anyway
# 5. Index storage cost becomes high

In [59]:
# CREATE INDEX idx_status
# ON orders(status);

In [60]:
# Why BigQuery does not use traditional indexes like OLTP databases

# BigQuery is a distributed columnar analytics engine. It is designed for scanning and processing very large datasets in parallel, not for single-row transactional lookup like MySQL/Postgres.

# Instead of traditional B-tree-style indexes, BigQuery mainly uses:

In [61]:
# partitioning
# clustering
# columnar storage
# block pruning
# materialized views
# search indexes for search-style use cases
# vector indexes for vector search use cases

In [62]:
# BigQuery clustered tables organize data by clustering columns and can scan only relevant blocks when filters match clustering columns. Google calls this block pruning.  

# BigQuery also supports search indexes, but those are specialized indexes for search over unstructured text / semi-structured JSON and certain operators, not the same as traditional OLTP indexes used for every relational lookup.  

# BigQuery also supports combining partitioning and clustering for query optimization.  

# Interview answer

# In OLTP databases, indexes help fast point lookups, but they add write overhead and storage cost. BigQuery is different because it is a distributed columnar warehouse optimized for parallel scans. Instead of traditional indexes, I normally use partitioning to reduce date ranges, clustering to prune blocks, and materialized views or summary tables for repeated aggregations. BigQuery has search indexes and vector indexes for specialized use cases, but not traditional indexes as the main tuning mechanism.

In [63]:
# 4. Correlated vs non-correlated subqueries

# Non-correlated subquery

# A non-correlated subquery can run independently.

# Example:

In [ ]:
# SELECT *

# FROM orders

# WHERE amount > (

#     SELECT AVG(amount)

#     FROM orders

# );

In [64]:
# SELECT AVG(amount)

# FROM orders

In [65]:
# 5. CTEs vs temp tables vs subqueries

# Subquery

# A subquery is useful for small, one-time logic.

# Example:

In [67]:
result_df = spark.sql("""

SELECT

    region,

    total_revenue

FROM (

    SELECT

        region,

        SUM(amount) AS total_revenue

    FROM orders

    GROUP BY region

) x

WHERE total_revenue > 300

""")

In [68]:
result_df = spark.sql("""

SELECT

    region,

    total_revenue

FROM (

    SELECT

        region,

        SUM(amount) AS total_revenue

    FROM orders

    GROUP BY region

) x

WHERE total_revenue > 300

""")

In [70]:
result_df = spark.sql("""

SELECT

    region,

    total_revenue

FROM (

    SELECT

        region,

        SUM(amount) AS total_revenue

    FROM orders

    GROUP BY region

) x

WHERE total_revenue > 300

""").show()

+-------+-------------+
| region|total_revenue|
+-------+-------------+
| Phuket|          900|
|Bangkok|          600|
+-------+-------------+



In [71]:
result_df = spark.sql("""
SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.amount
FROM orders_skewed o
JOIN customers c
    ON o.customer_id = c.customer_id
""")

result_df.show()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `orders_skewed` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 7 pos 5;
'Project ['o.order_id, 'o.customer_id, 'c.customer_name, 'o.amount]
+- 'Join Inner, ('o.customer_id = 'c.customer_id)
   :- 'SubqueryAlias o
   :  +- 'UnresolvedRelation [orders_skewed], [], false
   +- SubqueryAlias c
      +- SubqueryAlias customers
         +- View (`customers`, [customer_id#420,customer_name#421,segment#422])
            +- LogicalRDD [customer_id#420, customer_name#421, segment#422], false


In [72]:
result_df = spark.sql("""
SELECT /*+ BROADCAST(c) */
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.amount
FROM orders_skewed o
JOIN customers c
    ON o.customer_id = c.customer_id
""")

result_df.show()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `orders_skewed` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 7 pos 5;
'Project ['o.order_id, 'o.customer_id, 'c.customer_name, 'o.amount]
+- 'Join Inner, ('o.customer_id = 'c.customer_id)
   :- 'SubqueryAlias o
   :  +- 'UnresolvedRelation [orders_skewed], [], false
   +- ResolvedHint (strategy=broadcast)
      +- SubqueryAlias c
         +- SubqueryAlias customers
            +- View (`customers`, [customer_id#420,customer_name#421,segment#422])
               +- LogicalRDD [customer_id#420, customer_name#421, segment#422], false


In [73]:
from pyspark.sql.functions import col, floor, rand

salted_orders_df = orders_df.withColumn(
    "salt",
    floor(rand() * 5).cast("int")
)

salted_orders_df.createOrReplaceTempView("salted_orders")

In [75]:
salt_values_data = [(0,), (1,), (2,), (3,), (4,)]
salt_values_df = spark.createDataFrame(salt_values_data, ["salt"])
salt_values_df.createOrReplaceTempView("salt_values")

salted_customers_df = spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    s.salt
FROM customers c
CROSS JOIN salt_values s
""")

salted_customers_df.createOrReplaceTempView("salted_customers")

In [76]:
result_df = spark.sql("""
SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.amount
FROM salted_orders o
JOIN salted_customers c
    ON o.customer_id = c.customer_id
   AND o.salt = c.salt
""")

result_df.show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `o`.`customer_id` cannot be resolved. Did you mean one of the following? [`c`.`customer_id`, `c`.`customer_name`, `o`.`order_id`, `o`.`status`, `o`.`amount`].; line 9 pos 7;
'Project ['o.order_id, 'o.customer_id, 'c.customer_name, 'o.amount]
+- 'Join Inner, (('o.customer_id = customer_id#420) AND (cast(salt#673 as bigint) = salt#679L))
   :- SubqueryAlias o
   :  +- SubqueryAlias salted_orders
   :     +- View (`salted_orders`, [order_id#627L,region#628,status#629,amount#630L,salt#673])
   :        +- Project [order_id#627L, region#628, status#629, amount#630L, cast(FLOOR((rand(5171669480705020899) * cast(5 as double))) as int) AS salt#673]
   :           +- LogicalRDD [order_id#627L, region#628, status#629, amount#630L], false
   +- SubqueryAlias c
      +- SubqueryAlias salted_customers
         +- View (`salted_customers`, [customer_id#420,customer_name#421,salt#679L])
            +- Proj

In [77]:
orders_data = [
    (1, "C001", 100),
    (2, "C001", 200),
    (3, "C001", 300),
    (4, "C001", 400),
    (5, "C001", 500),
    (6, "C001", 600),
    (7, "C001", 700),
    (8, "C001", 800),
    (9, "C002", 100),
    (10, "C003", 150),
    (11, "C004", 200)
]

orders_columns = ["order_id", "customer_id", "amount"]

orders_df = spark.createDataFrame(orders_data, orders_columns)
orders_df.createOrReplaceTempView("orders_skewed")


customers_data = [
    ("C001", "Big Customer"),
    ("C002", "Small Customer 2"),
    ("C003", "Small Customer 3"),
    ("C004", "Small Customer 4")
]

customers_columns = ["customer_id", "customer_name"]

customers_df = spark.createDataFrame(customers_data, customers_columns)
customers_df.createOrReplaceTempView("customers")

In [79]:
runsql(

    """
    Select * from customers
    """
)

+-----------+----------------+
|customer_id|customer_name   |
+-----------+----------------+
|C001       |Big Customer    |
|C002       |Small Customer 2|
|C003       |Small Customer 3|
|C004       |Small Customer 4|
+-----------+----------------+



In [80]:
spark.sql("""
SELECT
    customer_id,
    COUNT(*) AS row_count
FROM orders_skewed
GROUP BY customer_id
ORDER BY row_count DESC
""").show()

+-----------+---------+
|customer_id|row_count|
+-----------+---------+
|       C001|        8|
|       C003|        1|
|       C004|        1|
|       C002|        1|
+-----------+---------+



In [82]:
spark.sql("""
SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.amount
FROM orders_skewed o
JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY o.order_id
""").show()

+--------+-----------+----------------+------+
|order_id|customer_id|   customer_name|amount|
+--------+-----------+----------------+------+
|       1|       C001|    Big Customer|   100|
|       2|       C001|    Big Customer|   200|
|       3|       C001|    Big Customer|   300|
|       4|       C001|    Big Customer|   400|
|       5|       C001|    Big Customer|   500|
|       6|       C001|    Big Customer|   600|
|       7|       C001|    Big Customer|   700|
|       8|       C001|    Big Customer|   800|
|       9|       C002|Small Customer 2|   100|
|      10|       C003|Small Customer 3|   150|
|      11|       C004|Small Customer 4|   200|
+--------+-----------+----------------+------+



In [83]:
salted_join_df = spark.sql("""
WITH salted_orders AS (
    SELECT
        order_id,
        customer_id,
        amount,
        CAST(FLOOR(RAND() * 5) AS INT) AS salt
    FROM orders_skewed
),

salt_values AS (
    SELECT 0 AS salt
    UNION ALL SELECT 1
    UNION ALL SELECT 2
    UNION ALL SELECT 3
    UNION ALL SELECT 4
),

salted_customers AS (
    SELECT
        c.customer_id,
        c.customer_name,
        s.salt
    FROM customers c
    CROSS JOIN salt_values s
)

SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.amount,
    o.salt
FROM salted_orders o
JOIN salted_customers c
    ON o.customer_id = c.customer_id
   AND o.salt = c.salt
ORDER BY o.order_id
""")

salted_join_df.show()

+--------+-----------+----------------+------+----+
|order_id|customer_id|   customer_name|amount|salt|
+--------+-----------+----------------+------+----+
|       1|       C001|    Big Customer|   100|   0|
|       2|       C001|    Big Customer|   200|   3|
|       3|       C001|    Big Customer|   300|   3|
|       4|       C001|    Big Customer|   400|   3|
|       5|       C001|    Big Customer|   500|   0|
|       6|       C001|    Big Customer|   600|   1|
|       7|       C001|    Big Customer|   700|   4|
|       8|       C001|    Big Customer|   800|   1|
|       9|       C002|Small Customer 2|   100|   1|
|      10|       C003|Small Customer 3|   150|   3|
|      11|       C004|Small Customer 4|   200|   3|
+--------+-----------+----------------+------+----+



In [84]:
salted_join_df = spark.sql("""
WITH salted_orders AS (
    SELECT
        order_id,
        customer_id,
        amount,
        CASE
            WHEN customer_id = 'C001'
            THEN CAST(FLOOR(RAND() * 5) AS INT)
            ELSE 0
        END AS salt
    FROM orders_skewed
),

salt_values AS (
    SELECT 0 AS salt
    UNION ALL SELECT 1
    UNION ALL SELECT 2
    UNION ALL SELECT 3
    UNION ALL SELECT 4
),

salted_customers AS (
    SELECT
        c.customer_id,
        c.customer_name,
        s.salt
    FROM customers c
    CROSS JOIN salt_values s
    WHERE c.customer_id = 'C001'

    UNION ALL

    SELECT
        customer_id,
        customer_name,
        0 AS salt
    FROM customers
    WHERE customer_id <> 'C001'
)

SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.amount,
    o.salt
FROM salted_orders o
JOIN salted_customers c
    ON o.customer_id = c.customer_id
   AND o.salt = c.salt
ORDER BY o.order_id
""")

salted_join_df.show()

+--------+-----------+----------------+------+----+
|order_id|customer_id|   customer_name|amount|salt|
+--------+-----------+----------------+------+----+
|       1|       C001|    Big Customer|   100|   2|
|       2|       C001|    Big Customer|   200|   3|
|       3|       C001|    Big Customer|   300|   1|
|       4|       C001|    Big Customer|   400|   1|
|       5|       C001|    Big Customer|   500|   1|
|       6|       C001|    Big Customer|   600|   1|
|       7|       C001|    Big Customer|   700|   2|
|       8|       C001|    Big Customer|   800|   1|
|       9|       C002|Small Customer 2|   100|   0|
|      10|       C003|Small Customer 3|   150|   0|
|      11|       C004|Small Customer 4|   200|   0|
+--------+-----------+----------------+------+----+



In [94]:
result_df = spark.sql("""
WITH salted_orders AS (
    SELECT
        order_id,
        customer_id,
        amount,
        cast(floor(RAND()*5) as int) ,
        CAST(FLOOR(RAND() * 5) AS INT) AS salt
    FROM orders_skewed
)
select * from salted_orders

""").show()

+--------+-----------+------+--------------------------------+----+
|order_id|customer_id|amount|CAST(FLOOR((rand() * 5)) AS INT)|salt|
+--------+-----------+------+--------------------------------+----+
|       1|       C001|   100|                               4|   1|
|       2|       C001|   200|                               1|   0|
|       3|       C001|   300|                               0|   1|
|       4|       C001|   400|                               0|   0|
|       5|       C001|   500|                               2|   0|
|       6|       C001|   600|                               0|   0|
|       7|       C001|   700|                               1|   4|
|       8|       C001|   800|                               3|   3|
|       9|       C002|   100|                               3|   3|
|      10|       C003|   150|                               2|   0|
|      11|       C004|   200|                               2|   4|
+--------+-----------+------+-------------------

In [99]:
salted_join_df = spark.sql("""
WITH salted_orders AS (
    SELECT
        order_id,
        customer_id,
        amount,
        CASE
            WHEN customer_id = 'C001'
            THEN CAST(FLOOR(RAND() * 5) AS INT)
            ELSE 0
        END AS salt
    FROM orders_skewed
),

salt_values AS (
    SELECT 0 AS salt
    UNION ALL SELECT 1
    UNION ALL SELECT 2
    UNION ALL SELECT 3
    UNION ALL SELECT 4
),
salted_customers AS (
    SELECT
        c.customer_id,
        c.customer_name,
        s.salt
    FROM customers c
    CROSS JOIN salt_values s
        WHERE c.customer_id = 'C001'
)
select * from salted_customers

""")

salted_join_df.show()

+-----------+-------------+----+
|customer_id|customer_name|salt|
+-----------+-------------+----+
|       C001| Big Customer|   0|
|       C001| Big Customer|   1|
|       C001| Big Customer|   2|
|       C001| Big Customer|   3|
|       C001| Big Customer|   4|
+-----------+-------------+----+



In [102]:
spark.sql("""
SELECT
    customer_id,
    COUNT(*) AS row_count
FROM orders_skewed
GROUP BY customer_id
ORDER BY row_count DESC
LIMIT 10
""").show()

+-----------+---------+
|customer_id|row_count|
+-----------+---------+
|       C001|        8|
|       C003|        1|
|       C004|        1|
|       C002|        1|
+-----------+---------+



In [113]:
orders_data = [
    (1, "C001", 100),
    (2, "C001", 200),
    (3, "C001", 300),
    (4, "C001", 400),
    (5, "C001", 500),
    (6, "C002", 100),
    (7, "C003", 150),
    (8, "C004", 200)
]

orders_columns = ["order_id", "customer_id", "amount"]

orders_df = spark.createDataFrame(orders_data, orders_columns)
orders_df.createOrReplaceTempView("orders")

In [115]:
spark.sql("""
SELECT
    customer_id,
    COUNT(*) AS row_count
FROM orders
GROUP BY customer_id
ORDER BY row_count DESC
""").show()

+-----------+---------+
|customer_id|row_count|
+-----------+---------+
|       C001|        5|
|       C003|        1|
|       C004|        1|
|       C002|        1|
+-----------+---------+



In [119]:
spark.sql("""
WITH salted_orders AS (
    SELECT
        order_id,
        customer_id,
        amount,
        HASH(order_id) as hash,
        CASE
            WHEN customer_id = 'C001'
            THEN PMOD(HASH(order_id), 5)
            ELSE 0
        END AS salt
    FROM orders
)

SELECT
    customer_id,
    salt,
    COUNT(*) AS row_count,
    SUM(amount) AS total_amount
FROM salted_orders
GROUP BY
    customer_id,
    salt
ORDER BY
    customer_id,
    salt
""").show()

+-----------+----+---------+------------+
|customer_id|salt|row_count|total_amount|
+-----------+----+---------+------------+
|       C001|   0|        1|         400|
|       C001|   2|        1|         300|
|       C001|   3|        2|         700|
|       C001|   4|        1|         100|
|       C002|   0|        1|         100|
|       C003|   0|        1|         150|
|       C004|   0|        1|         200|
+-----------+----+---------+------------+



In [133]:
spark.sql("""
WITH salted_orders AS (
    SELECT
        order_id,
        customer_id,
        amount,
        int(rand()*5)  as salt1,
        CASE
            WHEN customer_id = 'C001'
            THEN PMOD(HASH(order_id), 5)
            ELSE 0
        END AS salt
    FROM orders
),
 total as  (
SELECT
customer_id,
    salt1,
    COUNT(*) AS row_count,
    SUM(amount) AS total_amount
FROM salted_orders
GROUP BY
    customer_id,salt1
    )
    select SUM(total_amount),customer_id from total group by customer_id
    
    
""").show()

+-----------------+-----------+
|sum(total_amount)|customer_id|
+-----------------+-----------+
|              150|       C003|
|              200|       C004|
|             1500|       C001|
|              100|       C002|
+-----------------+-----------+

